# Optional Project - Colab Part4 Training and Generation

Runs Part 4: continue training the best model, generate unconditional and prefix-conditioned SVG samples, render them, evaluate validity metrics, and sync checkpoints/results to Google Drive.

In [1]:
# ===== User config =====
REPO_URL = "https://github.com/Peng-y-x/optionalproject.git"
REPO_DIR = "/content/optionalproject"
REPO_BRANCH = "run"
DATA_CONFIG = "configs/data.yaml"
PART4_CONFIG = "configs/part4_best.yaml"
GEN_CONFIG = "configs/part4_generation.yaml"
TEMP_GROUP_GEN_CONFIG = "configs/part4_generation_temperature_groups.yaml"
BEST_LR_JSON = "outputs/part3_mup_lr_sweep/best_lr.json"
DRIVE_BEST_LR_JSON = "/content/drive/MyDrive/svg-scaling/part3_mup_lr_sweep/best_lr.json"
DRIVE_ROOT = "/content/drive/MyDrive/svg-scaling"
PART4_RUN_DIR = "part4_best_v2"
PART4_SAMPLES_DIR = "part4_samples_v2"
PART4_EVAL_DIR = "part4_eval_v2"
MAX_EPOCHS = 3
# Optional cap for limited Colab sessions; set to 0 for full epochs.
MAX_TRAIN_TOKENS_PER_EPOCH = 0


In [4]:
# 1) Clone repo and checkout branch
import os
if not os.path.exists(REPO_DIR):
    !git clone --branch {REPO_BRANCH} --single-branch {REPO_URL} {REPO_DIR}
else:
    %cd $REPO_DIR
    !git fetch origin {REPO_BRANCH}
    !git checkout {REPO_BRANCH}
    !git pull --ff-only origin {REPO_BRANCH}


/content/optionalproject
From https://github.com/Peng-y-x/optionalproject
 * branch            run        -> FETCH_HEAD
Already on 'run'
Your branch is up to date with 'origin/run'.
From https://github.com/Peng-y-x/optionalproject
 * branch            run        -> FETCH_HEAD
Already up to date.


In [5]:
# 2) Mount Google Drive for resumable Part 4 checkpoints and final artifacts
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p {DRIVE_ROOT}/{PART4_RUN_DIR} {DRIVE_ROOT}/{PART4_SAMPLES_DIR} {DRIVE_ROOT}/{PART4_EVAL_DIR} {DRIVE_ROOT}/tokenizer


Mounted at /content/drive


In [6]:
# 3) Install system + Python dependencies
!apt-get update -y
!apt-get install -y libcairo2 libcairo2-dev libffi-dev
!python -m pip install -q -r {REPO_DIR}/requirements.txt


Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]               
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:6 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:7 https://cli.github.com/packages stable/main amd64 Packages [356 B]       
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease    
Hit:9 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]      
Get:11 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [61.6 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Package

In [ ]:
# 4) HF auth from Colab Keys (key name must be HF_TOKEN)
import os
from google.colab import userdata
token = userdata.get('HF_TOKEN')
if token:
    os.environ['HF_TOKEN'] = token
print('has_hf_token:', bool(os.getenv('HF_TOKEN')))


In [8]:
# 5) GPU sanity check
import torch
print('torch', torch.__version__)
print('cuda_available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))


torch 2.10.0+cu128
cuda_available True
gpu Tesla T4


In [9]:
# 6) Resolve and verify the tokenizer for generation.
# Important: do not download Zala0429/svg-scaling-tokenizer-v1 here; it was verified not to decode the tokenized dataset correctly.
# Use the tokenizer artifact that matches Zala0429/svg-scaling-v2-tokenized. The alignment check below fails fast if it is wrong.
%cd $REPO_DIR
from pathlib import Path
import shutil, yaml, subprocess

cfg = yaml.safe_load(Path(DATA_CONFIG).read_text(encoding='utf-8'))
tok_path = Path(cfg['tokenization']['output_dir']) / 'tokenizer.json'
candidate_paths = [
    tok_path,
    Path(DRIVE_ROOT) / 'tokenizer_matching/tokenizer.json',
    Path(DRIVE_ROOT) / 'tokenizer/tokenizer.json',
]
existing = next((p for p in candidate_paths if p.exists()), None)
if existing is None:
    raise FileNotFoundError('No tokenizer.json found. Restore the exact tokenizer used to create Zala0429/svg-scaling-v2-tokenized.')
tok_path.parent.mkdir(parents=True, exist_ok=True)
if existing != tok_path:
    shutil.copy2(existing, tok_path)
print('using tokenizer:', tok_path)
subprocess.run(['python', 'scripts/check_tokenizer_alignment.py', '--tokenizer-path', str(tok_path), '--num-samples', '5'], check=True)


/content/optionalproject
using tokenizer: data/processed/v1-clean-rawsplit/tokenizer/tokenizer.json


CompletedProcess(args=['python', 'scripts/check_tokenizer_alignment.py', '--tokenizer-path', 'data/processed/v1-clean-rawsplit/tokenizer/tokenizer.json', '--num-samples', '5'], returncode=0)

In [8]:
# 7) Resolve Part 3 best LR and initialization checkpoint.
# The Part 4 run writes to its own Drive directory and resumes from there after disconnects.
%cd $REPO_DIR
from pathlib import Path
import shutil

if not Path(BEST_LR_JSON).exists() and Path(DRIVE_BEST_LR_JSON).exists():
    Path(BEST_LR_JSON).parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE_BEST_LR_JSON, BEST_LR_JSON)

init_candidates = [
    Path(DRIVE_ROOT) / 'part3_mup/xl_mup/checkpoints/best.pt',
    Path(DRIVE_ROOT) / 'part3_mup/xl_mup/checkpoints/latest.pt',
    Path(DRIVE_ROOT) / 'part2_v2/xl/checkpoints/best.pt',
    Path(DRIVE_ROOT) / 'part2_v2/xl/checkpoints/latest.pt',
]
INIT_FROM = next((str(p) for p in init_candidates if p.exists()), '')
print('best_lr_json exists:', Path(BEST_LR_JSON).exists())
print('init_from:', INIT_FROM or '(none; train from scratch)')


/content/optionalproject
best_lr_json exists: True
init_from: /content/drive/MyDrive/svg-scaling/part3_mup/xl_mup/checkpoints/best.pt


In [9]:
# 8) Continue training best model for Part 4 v2.
# If Colab disconnects, rerun this cell; it resumes from Drive PART4_RUN_DIR.
%cd $REPO_DIR
import subprocess
from pathlib import Path

cmd = ['python', 'scripts/run_part4_train.py', '--config', PART4_CONFIG, '--parameterization', 'mup', '--max-epochs', str(MAX_EPOCHS)]
if Path(BEST_LR_JSON).exists():
    cmd += ['--best-lr-json', BEST_LR_JSON]
else:
    cmd += ['--learning-rate', '0.0006']
if INIT_FROM:
    cmd += ['--init-from', INIT_FROM]
if MAX_TRAIN_TOKENS_PER_EPOCH and MAX_TRAIN_TOKENS_PER_EPOCH > 0:
    cmd += ['--max-train-tokens-per-epoch', str(MAX_TRAIN_TOKENS_PER_EPOCH)]
print(' '.join(cmd))
subprocess.run(cmd, check=True)


/content/optionalproject
python scripts/run_part4_train.py --config configs/part4_best.yaml --parameterization mup --max-epochs 3 --best-lr-json outputs/part3_mup_lr_sweep/best_lr.json --init-from /content/drive/MyDrive/svg-scaling/part3_mup/xl_mup/checkpoints/best.pt


CompletedProcess(args=['python', 'scripts/run_part4_train.py', '--config', 'configs/part4_best.yaml', '--parameterization', 'mup', '--max-epochs', '3', '--best-lr-json', 'outputs/part3_mup_lr_sweep/best_lr.json', '--init-from', '/content/drive/MyDrive/svg-scaling/part3_mup/xl_mup/checkpoints/best.pt'], returncode=0)

In [21]:
# 9) Inspect Part 4 v2 training output
%cd $REPO_DIR
from pathlib import Path
import json
run_dir = Path('outputs') / PART4_RUN_DIR / 'best_mup_xl'
if not run_dir.exists():
    run_dir = Path(DRIVE_ROOT) / PART4_RUN_DIR / 'best_mup_xl'
print('run_dir:', run_dir)
for p in [run_dir / 'summary.json', run_dir / 'final_metrics.json']:
    print('\n', p, p.exists())
    if p.exists():
        print(json.dumps(json.loads(p.read_text()), indent=2)[:2000])


/content/optionalproject
run_dir: /content/drive/MyDrive/svg-scaling/part4_best_v2/best_mup_xl

 /content/drive/MyDrive/svg-scaling/part4_best_v2/best_mup_xl/summary.json False

 /content/drive/MyDrive/svg-scaling/part4_best_v2/best_mup_xl/final_metrics.json False


In [22]:
# 10) Generate v2 unconditional and prefix-conditioned SVG samples, render PNGs, and sync to Drive.
%cd $REPO_DIR
from pathlib import Path
import subprocess

ckpt_candidates = [
    Path('outputs') / PART4_RUN_DIR / 'best_mup_xl/checkpoints/best.pt',
    Path('outputs') / PART4_RUN_DIR / 'best_mup_xl/checkpoints/latest.pt',
    Path(DRIVE_ROOT) / PART4_RUN_DIR / 'best_mup_xl/checkpoints/best.pt',
    Path(DRIVE_ROOT) / PART4_RUN_DIR / 'best_mup_xl/checkpoints/latest.pt',
    Path(DRIVE_ROOT) / 'part4_best/best_mup_xl/checkpoints/best.pt',
    Path(DRIVE_ROOT) / 'part4_best/best_mup_xl/checkpoints/latest.pt',
]
PART4_CKPT = next(str(p) for p in ckpt_candidates if p.exists())
TOKENIZER_PATH = 'data/processed/v1-clean-rawsplit/tokenizer/tokenizer.json'
cmd = [
    'python', 'scripts/run_generate.py',
    '--config', GEN_CONFIG,
    '--checkpoint-path', PART4_CKPT,
    '--tokenizer-path', TOKENIZER_PATH,
    '--tokenizer-hf-repo-id', TOKENIZER_HF_REPO,
    '--output-dir', f'outputs/{PART4_SAMPLES_DIR}',
    '--drive-output-dir', f'{DRIVE_ROOT}/{PART4_SAMPLES_DIR}',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)


/content/optionalproject
python scripts/run_generate.py --config configs/part4_generation.yaml --checkpoint-path /content/drive/MyDrive/svg-scaling/part4_best/best_mup_xl/checkpoints/best.pt --tokenizer-path data/processed/v1-clean-rawsplit/tokenizer/tokenizer.json --tokenizer-hf-repo-id Zala0429/svg-scaling-tokenizer-v1 --output-dir outputs/part4_samples_v2 --drive-output-dir /content/drive/MyDrive/svg-scaling/part4_samples_v2


CompletedProcess(args=['python', 'scripts/run_generate.py', '--config', 'configs/part4_generation.yaml', '--checkpoint-path', '/content/drive/MyDrive/svg-scaling/part4_best/best_mup_xl/checkpoints/best.pt', '--tokenizer-path', 'data/processed/v1-clean-rawsplit/tokenizer/tokenizer.json', '--tokenizer-hf-repo-id', 'Zala0429/svg-scaling-tokenizer-v1', '--output-dir', 'outputs/part4_samples_v2', '--drive-output-dir', '/content/drive/MyDrive/svg-scaling/part4_samples_v2'], returncode=0)

In [23]:
# 11) Evaluate Part 4 v2: test perplexity + XML/render/structural validity for generated samples.
%cd $REPO_DIR
import subprocess
cmd = [
    'python', 'scripts/run_eval.py',
    '--train-config', PART4_CONFIG,
    '--checkpoint-path', PART4_CKPT,
    '--samples-jsonl', f'outputs/{PART4_SAMPLES_DIR}/samples.jsonl',
    '--output-dir', f'outputs/{PART4_EVAL_DIR}',
    '--drive-output-dir', f'{DRIVE_ROOT}/{PART4_EVAL_DIR}',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)


/content/optionalproject
python scripts/run_eval.py --train-config configs/part4_best.yaml --checkpoint-path /content/drive/MyDrive/svg-scaling/part4_best/best_mup_xl/checkpoints/best.pt --samples-jsonl outputs/part4_samples_v2/samples.jsonl --output-dir outputs/part4_eval_v2 --drive-output-dir /content/drive/MyDrive/svg-scaling/part4_eval_v2


CompletedProcess(args=['python', 'scripts/run_eval.py', '--train-config', 'configs/part4_best.yaml', '--checkpoint-path', '/content/drive/MyDrive/svg-scaling/part4_best/best_mup_xl/checkpoints/best.pt', '--samples-jsonl', 'outputs/part4_samples_v2/samples.jsonl', '--output-dir', 'outputs/part4_eval_v2', '--drive-output-dir', '/content/drive/MyDrive/svg-scaling/part4_eval_v2'], returncode=0)

In [24]:
# 12) Inspect final Part 4 v2 artifacts
%cd $REPO_DIR
from pathlib import Path
import json
for p in [
    Path('outputs') / PART4_SAMPLES_DIR / 'generation_summary.json',
    Path('outputs') / PART4_EVAL_DIR / 'part4_metrics.json',
    Path('outputs') / PART4_SAMPLES_DIR / 'generated_grid.png',
]:
    print('\n', p, p.exists())
    if p.suffix == '.json' and p.exists():
        print(json.dumps(json.loads(p.read_text()), indent=2)[:2000])
print('Drive results:', f'{DRIVE_ROOT}/{PART4_SAMPLES_DIR}', f'{DRIVE_ROOT}/{PART4_EVAL_DIR}')


/content/optionalproject

 outputs/part4_samples_v2/generation_summary.json True
{
  "checkpoint_path": "/content/drive/MyDrive/svg-scaling/part4_best/best_mup_xl/checkpoints/best.pt",
  "tokenizer_path": "data/processed/v1-clean-rawsplit/tokenizer/tokenizer.json",
  "tokenizer_hf_repo_id": "Zala0429/svg-scaling-tokenizer-v1",
  "stop_text": "</svg>",
  "num_samples": 45,
  "num_rendered": 3,
  "checkpoint_config": {
    "seed": 42,
    "run": {
      "name": "best_mup_xl",
      "output_dir": "outputs/part4_best",
      "drive_output_dir": "/content/drive/MyDrive/svg-scaling/part4_best",
      "resume": "auto",
      "save_latest_only": true,
      "device": "cuda",
      "precision": "bf16",
      "init_from_checkpoint": "/content/drive/MyDrive/svg-scaling/part3_mup/xl_mup/checkpoints/best.pt"
    },
    "data": {
      "source": "hf",
      "hf_repo_id": "Zala0429/svg-scaling-v2-tokenized",
      "train_total_tokens": 284601303,
      "mean_seq_len": 668
    },
    "model": {
      

In [10]:
# Optional: rerun Part 4 generation as temperature-controlled groups.
# 3 groups: each has 20 unconditional samples + 5 prefix completions from complete reference SVGs.
# This does not retrain the model and writes to a new Drive folder.
%cd $REPO_DIR
from pathlib import Path
import subprocess

ckpt_candidates = [
    Path(DRIVE_ROOT) / 'part4_best/best_mup_xl/checkpoints/best.pt',
    Path(DRIVE_ROOT) / 'part4_best/best_mup_xl/checkpoints/latest.pt',
    Path('outputs/part4_best/best_mup_xl/checkpoints/best.pt'),
    Path('outputs/part4_best/best_mup_xl/checkpoints/latest.pt'),
]
GROUP_CKPT = next(str(p) for p in ckpt_candidates if p.exists())
TOKENIZER_PATH = 'data/processed/v1-clean-rawsplit/tokenizer/tokenizer.json'
cmd = [
    'python', 'scripts/run_generate.py',
    '--config', TEMP_GROUP_GEN_CONFIG,
    '--checkpoint-path', GROUP_CKPT,
    '--tokenizer-path', TOKENIZER_PATH,
    '--output-dir', 'outputs/part4_generation_temperature_groups',
    '--drive-output-dir', f'{DRIVE_ROOT}/part4_generation_temperature_groups',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)
print('Drive output:', f'{DRIVE_ROOT}/part4_generation_temperature_groups')


/content/optionalproject
python scripts/run_generate.py --config configs/part4_generation_temperature_groups.yaml --checkpoint-path /content/drive/MyDrive/svg-scaling/part4_best/best_mup_xl/checkpoints/best.pt --tokenizer-path data/processed/v1-clean-rawsplit/tokenizer/tokenizer.json --output-dir outputs/part4_generation_temperature_groups --drive-output-dir /content/drive/MyDrive/svg-scaling/part4_generation_temperature_groups
Drive output: /content/drive/MyDrive/svg-scaling/part4_generation_temperature_groups


In [11]:
# Evaluate the temperature-group generation run.
%cd $REPO_DIR
import subprocess
cmd = [
    'python', 'scripts/run_eval.py',
    '--train-config', PART4_CONFIG,
    '--checkpoint-path', GROUP_CKPT,
    '--samples-jsonl', 'outputs/part4_generation_temperature_groups/samples.jsonl',
    '--output-dir', 'outputs/part4_generation_temperature_groups_eval',
    '--drive-output-dir', f'{DRIVE_ROOT}/part4_generation_temperature_groups_eval',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)
print('Drive eval output:', f'{DRIVE_ROOT}/part4_generation_temperature_groups_eval')


/content/optionalproject
python scripts/run_eval.py --train-config configs/part4_best.yaml --checkpoint-path /content/drive/MyDrive/svg-scaling/part4_best/best_mup_xl/checkpoints/best.pt --samples-jsonl outputs/part4_generation_temperature_groups/samples.jsonl --output-dir outputs/part4_generation_temperature_groups_eval --drive-output-dir /content/drive/MyDrive/svg-scaling/part4_generation_temperature_groups_eval
Drive eval output: /content/drive/MyDrive/svg-scaling/part4_generation_temperature_groups_eval


In [12]:
# Summarize success/failure by temperature group and sample kind.
from pathlib import Path
import json
samples_path = Path("outputs/part4_generation_temperature_groups/samples.jsonl")
eval_path = Path("outputs/part4_generation_temperature_groups_eval/sample_validity.jsonl")
rows = [json.loads(line) for line in samples_path.read_text(encoding="utf-8").splitlines() if line.strip()]
valid = {json.loads(line)["id"]: json.loads(line) for line in eval_path.read_text(encoding="utf-8").splitlines() if line.strip()}
for group in sorted({r.get("group") for r in rows}):
    print("\nGROUP", group)
    for kind in ["unconditional", "prefix"]:
        ids = [r["id"] for r in rows if r.get("group") == group and r.get("kind") == kind]
        ok = sum(1 for sid in ids if valid.get(sid, {}).get("render_valid"))
        print(kind, ok, "/", len(ids), "render valid")
print("\nGenerated grid:", Path("outputs/part4_generation_temperature_groups/generated_grid.png").exists())



GROUP temp_0_5
unconditional 5 / 20 render valid
prefix 0 / 5 render valid

GROUP temp_0_8
unconditional 5 / 20 render valid
prefix 0 / 5 render valid

GROUP temp_1_0
unconditional 10 / 20 render valid
prefix 1 / 5 render valid

Generated grid: True
